In [0]:
# Databricks 노트북 - 산책로/공원 경사도·토양 전처리
# silver.trail_features 테이블 생성 → PostgreSQL 적재
# 실행 전: /mnt/raw, /mnt/silver 마운트 확인

%pip install geopandas

In [0]:
import geopandas as gpd
import pandas as pd
import json
from pyspark.sql.functions import col, lit, when, udf
from pyspark.sql.types import StringType, FloatType

# 입력 파일 경로 (Blob 마운트 기준)
SLOPE_SHP   = "/dbfs/mnt/raw/vworld/slope/gangnam_songpa_gangdong_soilslope.shp"
SOIL_SHP    = "/dbfs/mnt/raw/vworld/soil_type/gangnam_songpa_gangdong_soil_type.shp"
TRAIL_CSV   = "/dbfs/mnt/raw/walkwayData/target_major_trails.csv"
PARK_CSV    = "/dbfs/mnt/raw/walkwayData/target_major_parks.csv"

print("✅ 경로 설정 완료")
print(f"  경사도 SHP : {SLOPE_SHP}")
print(f"  토양   SHP : {SOIL_SHP}")
print(f"  산책로 CSV : {TRAIL_CSV}")
print(f"  공원   CSV : {PARK_CSV}")

✅ 경로 설정 완료
  경사도 SHP : /dbfs/mnt/raw/vworld/slope/gangnam_songpa_gangdong_soilslope.shp
  토양   SHP : /dbfs/mnt/raw/vworld/soil_type/gangnam_songpa_gangdong_soil_type.shp
  산책로 CSV : /dbfs/mnt/raw/walkwayData/target_major_trails.csv
  공원   CSV : /dbfs/mnt/raw/walkwayData/target_major_parks.csv


In [0]:
# SHP 파일 로드 및 좌표계 변환
WGS84 = "EPSG:4326"

print("1. 경사도 SHP 로드...")
gdf_slope = gpd.read_file(SLOPE_SHP, encoding='cp949')
if gdf_slope.crs is None:
    gdf_slope = gdf_slope.set_crs(epsg=5181)
gdf_slope = gdf_slope.to_crs(WGS84)
print(f"   경사도 폴리곤 {len(gdf_slope)}개 로드 완료, CRS → WGS84")

print("2. 토양 SHP 로드...")
gdf_soil = gpd.read_file(SOIL_SHP, encoding='utf-8')
if gdf_soil.crs is None:
    gdf_soil = gdf_soil.set_crs(epsg=5181)
gdf_soil = gdf_soil.to_crs(WGS84)
print(f"   토양 폴리곤 {len(gdf_soil)}개 로드 완료, CRS → WGS84")

1. 경사도 SHP 로드...
   경사도 폴리곤 303개 로드 완료, CRS → WGS84
2. 토양 SHP 로드...
   토양 폴리곤 171개 로드 완료, CRS → WGS84


/local_disk0/.ephemeral_nfs/envs/pythonEnv-3a9a0703-d3fb-4035-842e-804bce2c973b/lib/python3.11/site-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp949 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(


In [0]:
# 경사도 수치 변환 함수
import re
 
SLOPE_MIDPOINTS = {
    "0-2%": 1.0,   "0~2%": 1.0,
    "2-7%": 4.5,   "2~7%": 4.5,
    "0-7%": 3.5,   "0~7%": 3.5,
    "7-15%": 11.0, "7~15%": 11.0,
    "15-30%": 22.5,"15~30%": 22.5,
    "30-60%": 45.0,"30~60%": 45.0,
    "60%이상": 70.0,"60% 이상": 70.0,
    "60-100%": 80.0,"60~100%": 80.0
}
 
def val_to_num(slope_val):
    for k, v in SLOPE_MIDPOINTS.items():
        if k in str(slope_val):
            return v
    nums = re.findall(r'\d+', str(slope_val))
    if len(nums) == 2:
        return (float(nums[0]) + float(nums[1])) / 2.0
    return None
 
def categorize_slope(avg_pct):
    if avg_pct is None:
        return "정보 없음"
    if avg_pct < 3.0:
        return "평지"
    elif avg_pct < 8.0:
        return "완만"
    else:
        return "경사"
 
print("✅ 경사도 변환 함수 정의 완료")
 

✅ 경사도 변환 함수 정의 완료


In [0]:
# 산책로 CSV 로드 및 공간 결합
print("3. 산책로 CSV 로드...")

trail_df = pd.read_csv(TRAIL_CSV, encoding='utf-8')
print(f"   컬럼 목록: {trail_df.columns.tolist()}")
print(f"   행 수: {len(trail_df)}")
trail_df.head(3)

3. 산책로 CSV 로드...
   컬럼 목록: ['TRL_ID', 'TRL_NM', 'TRL_CD', 'CTPRV_CD', 'SGNG_CD', 'EMNDN_CD', 'LGDNG_LI_CD', 'MLTNO_YN', 'LTNO_ORN', 'LTNO_VCN', 'PNTM_XCRD', 'PNTM_YCRD', 'TRMNA_XCRD', 'TRMNA_YCRD']
   행 수: 20


,TRL_ID,TRL_NM,TRL_CD,CTPRV_CD,SGNG_CD,EMNDN_CD,LGDNG_LI_CD,MLTNO_YN,LTNO_ORN,LTNO_VCN,PNTM_XCRD,PNTM_YCRD,TRMNA_XCRD,TRMNA_YCRD
0,C003,고덕산 산책길,2,11,740,107,0,2,41,3,127.146467,37.560926,127.145489,37.564160
1,C009,구룡산 나들길,2,11,680,118,0,1,468,0,127.048030,37.483104,127.061203,37.468921
2,C014,대모산 나들길,2,11,680,112,0,2,46,43,127.091607,37.477625,127.074204,37.478188


In [0]:
print(trail_df[['TRL_NM', 'PNTM_XCRD', 'PNTM_YCRD']].head(5))

      TRL_NM   PNTM_XCRD  PNTM_YCRD
0    고덕산 산책길  127.146467  37.560926
1    구룡산 나들길  127.048030  37.483104
2    대모산 나들길  127.091607  37.477625
3     선정릉나들길  127.058046  37.513966
4  일자산 숲 나들길  127.153721  37.539162


In [0]:
# 산책로 위경도 → GeoDataFrame 변환

LAT_COL = 'PNTM_YCRD'   # Y좌표 (위도에 해당)
LNG_COL = 'PNTM_XCRD'   # X좌표 (경도에 해당)
NAME_COL = 'TRL_NM'     # 산책로 명칭 (Trail Name)

trail_gdf = gpd.GeoDataFrame(
    trail_df,
    geometry=gpd.points_from_xy(trail_df[LNG_COL], trail_df[LAT_COL]),
    crs=WGS84
)
print(f"   산책로 GeoDataFrame 생성 완료: {len(trail_gdf)}개")

   산책로 GeoDataFrame 생성 완료: 20개


In [0]:
print("4. 산책로 × 경사도 sjoin...")

slope_col = 'SOILSLOPE' if 'SOILSLOPE' in gdf_slope.columns else gdf_slope.columns[1]
print(f"   경사도 컬럼: {slope_col}")

trail_slope = gpd.sjoin(
    trail_gdf[['geometry', NAME_COL]],
    gdf_slope[['geometry', slope_col]],
    how='left',
    predicate='within'
)

# 경사도 수치 변환 및 집계 (이름 기준 평균)
trail_slope['slope_num'] = trail_slope[slope_col].apply(val_to_num)
slope_summary = trail_slope.groupby(NAME_COL)['slope_num'].mean().reset_index()
slope_summary.columns = [NAME_COL, 'avg_slope']
slope_summary['slope_type'] = slope_summary['avg_slope'].apply(categorize_slope)

print(f"   경사도 매핑 완료: {len(slope_summary)}건")
print(slope_summary.head())


4. 산책로 × 경사도 sjoin...
   경사도 컬럼: SOILSLOPE
   경사도 매핑 완료: 20건
          TRL_NM  avg_slope slope_type
0     7코스-일자산 코스       22.5         경사
1   8코스-장지 탄천 코스        1.0         평지
2  9코스-대모 구룡산 코스        1.0         평지
3        고덕산 산책길       11.0         경사
4        고덕산 자락길        1.0         평지


In [0]:
print("5. 산책로 × 토양 sjoin...")

soil_col = 'DEEPSOIL' if 'DEEPSOIL' in gdf_soil.columns else gdf_soil.columns[1]
print(f"   토양 컬럼: {soil_col}")

trail_soil = gpd.sjoin(
    trail_gdf[['geometry', NAME_COL]],
    gdf_soil[['geometry', soil_col]],
    how='left',
    predicate='within'
)

from collections import Counter
def most_common(vals):
    vals = [v for v in vals if pd.notna(v)]
    return Counter(vals).most_common(1)[0][0] if vals else "정보 없음"

soil_summary = trail_soil.groupby(NAME_COL)[soil_col].apply(most_common).reset_index()
soil_summary.columns = [NAME_COL, 'soil_type']

print(f"   토양 매핑 완료: {len(soil_summary)}건")
print(soil_summary.head())


5. 산책로 × 토양 sjoin...
   토양 컬럼: DEEPSOIL
   토양 매핑 완료: 20건
          TRL_NM soil_type
0     7코스-일자산 코스       사양질
1   8코스-장지 탄천 코스       사양질
2  9코스-대모 구룡산 코스       사양질
3        고덕산 산책길       사양질
4        고덕산 자락길       사양질


In [0]:
print("6. 공원 CSV 로드 및 sjoin...")

park_df = pd.read_csv(PARK_CSV, encoding='utf-8')
print(f"   공원 컬럼: {park_df.columns.tolist()}")

PARK_LAT_COL  = 'Y좌표(WGS84)'
PARK_LNG_COL  = 'X좌표(WGS84)'
PARK_NAME_COL = '공원명'

park_gdf = gpd.GeoDataFrame(
    park_df,
    geometry=gpd.points_from_xy(park_df[PARK_LNG_COL], park_df[PARK_LAT_COL]),
    crs=WGS84
)

# 공원 경사도 sjoin
park_slope = gpd.sjoin(
    park_gdf[['geometry', PARK_NAME_COL]],
    gdf_slope[['geometry', slope_col]],
    how='left', predicate='within'
)
park_slope['slope_num'] = park_slope[slope_col].apply(val_to_num)
park_slope_sum = park_slope.groupby(PARK_NAME_COL)['slope_num'].mean().reset_index()
park_slope_sum.columns = [NAME_COL, 'avg_slope']
park_slope_sum['slope_type'] = park_slope_sum['avg_slope'].apply(categorize_slope)

# 공원 토양 sjoin
park_soil = gpd.sjoin(
    park_gdf[['geometry', PARK_NAME_COL]],
    gdf_soil[['geometry', soil_col]],
    how='left', predicate='within'
)
park_soil_sum = park_soil.groupby(PARK_NAME_COL)[soil_col].apply(most_common).reset_index()
park_soil_sum.columns = [NAME_COL, 'soil_type']

print(f"   공원 경사도 {len(park_slope_sum)}건, 토양 {len(park_soil_sum)}건 완료")


6. 공원 CSV 로드 및 sjoin...
   공원 컬럼: ['연번', '공원명', '공원개요', '면적', '개원일', '주요시설', '주요식물', '안내도', '오시는길', '이용시참고사항', '이미지', '지역', '공원주소', '관리부서', '전화번호', 'X좌표(GRS80TM)', 'Y좌표(GRS80TM)', 'X좌표(WGS84)', 'Y좌표(WGS84)', '바로가기']
   공원 경사도 21건, 토양 21건 완료


In [0]:
print("7. 전체 통합...")

# 산책로 + 공원 합치기
all_slope = pd.concat([slope_summary, park_slope_sum], ignore_index=True)
all_soil  = pd.concat([soil_summary,  park_soil_sum],  ignore_index=True)

# 최종 병합
final_df = all_slope.merge(all_soil, on=NAME_COL, how='outer')
final_df['avg_slope'] = final_df['avg_slope'].fillna(0.0)
final_df['slope_type'] = final_df['slope_type'].fillna("정보 없음")
final_df['soil_type']  = final_df['soil_type'].fillna("정보 없음")

print(f"   최종 행 수: {len(final_df)}")
print(final_df.head(10))


7. 전체 통합...
   최종 행 수: 41
          TRL_NM  avg_slope slope_type soil_type
0     7코스-일자산 코스       22.5         경사       사양질
1   8코스-장지 탄천 코스        1.0         평지       사양질
2  9코스-대모 구룡산 코스        1.0         평지       사양질
3        고덕산 산책길       11.0         경사       사양질
4        고덕산 자락길        1.0         평지       사양질
5           고덕천길        1.0         평지       사양질
6        구룡산 나들길        1.0         평지       사양질
7        대모산 나들길       11.0         경사       사양질
8            독정천        1.0         평지       사양질
9           망월천길        1.0         평지       사양질


In [0]:
print("8. Silver Delta Table 저장...")

spark_df = spark.createDataFrame(final_df[[NAME_COL, 'avg_slope', 'slope_type', 'soil_type']])
spark_df = spark_df.withColumnRenamed(NAME_COL, 'trail_name')

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.trail_features")

print(f"✅ silver.trail_features 저장 완료! ({spark_df.count()}건)")
spark_df.show(10)


8. Silver Delta Table 저장...
✅ silver.trail_features 저장 완료! (41건)
+----------------------+---------+----------+---------+
|            trail_name|avg_slope|slope_type|soil_type|
+----------------------+---------+----------+---------+
|     7코스-일자산 코스|     22.5|      경사|   사양질|
|  8코스-장지 탄천 코스|      1.0|      평지|   사양질|
|9코스-대모 구룡산 코스|      1.0|      평지|   사양질|
|         고덕산 산책길|     11.0|      경사|   사양질|
|         고덕산 자락길|      1.0|      평지|   사양질|
|              고덕천길|      1.0|      평지|   사양질|
|         구룡산 나들길|      1.0|      평지|   사양질|
|         대모산 나들길|     11.0|      경사|   사양질|
|                독정천|      1.0|      평지|   사양질|
|              망월천길|      1.0|      평지|   사양질|
+----------------------+---------+----------+---------+
only showing top 10 rows



In [0]:
print("9. PostgreSQL 적재...")

# 운영 환경에서는 Databricks Secret 또는 환경변수 사용 권장
jdbc_url = "<JDBC_URL>"
connection_properties = {
    "user": "<USERNAME>",
    "password": "<PASSWORD>",
    "driver": "org.postgresql.Driver"
}

spark_df.write.jdbc(
    url=jdbc_url,
    table="trail_features",
    mode="overwrite",
    properties=connection_properties
)

print("✅ PostgreSQL trail_features 적재 완료!")

# 검증
verify = spark.read.jdbc(
    url=jdbc_url,
    table="trail_features",
    properties=connection_properties
)
print(f"   PostgreSQL 행 수: {verify.count()}")
verify.show(5)

9. PostgreSQL 적재...
✅ PostgreSQL trail_features 적재 완료!
   PostgreSQL 행 수: 41
+----------------------+---------+----------+---------+
|            trail_name|avg_slope|slope_type|soil_type|
+----------------------+---------+----------+---------+
|          도곡근린공원|     22.5|      경사|   사양질|
|          도산근린공원|      4.5|      완만|   사양질|
|9코스-대모 구룡산 코스|      1.0|      평지|   사양질|
|         고덕산 산책길|     11.0|      경사|   사양질|
|         고덕산 자락길|      1.0|      평지|   사양질|
+----------------------+---------+----------+---------+
only showing top 5 rows

